# Un mini-GPT desde cero: constrúyelo y míralo pensar

**Facsímil 3 · Arquitecturas y modelos** — capítulos 1 a 4
(de los tokens a la atención, el bloque Transformer y el muestreo: todo junto, ejecutable).

En los capítulos del facsímil viste cada pieza de un Transformer por separado: la tokenización, los
*embeddings*, la atención con Q, K y V, la máscara causal, las conexiones residuales y la
normalización. Aquí las **ensamblas todas** en un GPT diminuto, lo **entrenas** sobre un texto en
español y lo ves **generar** letra a letra. No es una maqueta de mentira: es la misma arquitectura
de los modelos grandes, encogida hasta caber en este cuaderno.

Y, como en el [Transformer Explainer](https://poloclub.github.io/transformer-explainer/) de Georgia
Tech, podrás **mirar por dentro**: ver el mapa de atención de cada cabeza y la distribución con la
que el modelo elige la próxima letra.

### Qué vas a hacer
- **Tokenizar** un texto a nivel de carácter y convertirlo en números.
- Construir, pieza a pieza, un **bloque Transformer**: atención multi-cabeza causal, *feed-forward*, residual y *layernorm*.
- Ensamblar el **mini-GPT** completo y contar sus parámetros.
- Verlo generar **antes** de entrenar (ruido) y **después** (español reconocible).
- **Entrenarlo** y seguir la curva de pérdida en directo.
- **Mirar la atención**: qué letras miran a cuáles, con mapas de calor.
- Jugar con la **temperatura** y ver cómo cambia la distribución de la próxima letra.

### Cuánto cuesta
Unos 10-15 minutos. **Mejor con GPU** (en Colab: *Entorno de ejecución → Cambiar tipo de entorno → GPU*; es gratis).
En CPU también funciona, solo tarda algo más. Sin claves ni datos externos. Lee cada celda de texto antes de correr la de código.

> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-28 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

## 0. Preparación

Solo necesitamos **PyTorch** (para las redes) y **matplotlib** (para ver). En Colab ya vienen
instalados. Fijamos una semilla para que tus resultados sean reproducibles, y detectamos si hay GPU.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '· dispositivo:', device)
if device == 'cpu':
    print('Sugerencia: en Colab activa la GPU (Entorno de ejecución -> Cambiar tipo) para que entrene mucho más rápido.')

## 1. El texto que va a leer

Un modelo de lenguaje no sabe nada al nacer: todo lo que aprende sale del texto que le damos.
Aquí usamos un texto en español sobre la propia IA. Es pequeño a propósito, para que entrene rápido;
con más texto, el modelo generaría mejor. **Puedes cambiarlo por el tuyo** (un texto largo tuyo, las
letras de tus canciones favoritas, lo que quieras) y repetir el cuaderno: el modelo aprenderá *ese* estilo.

In [ ]:
texto = """La inteligencia artificial aprende del texto. Un modelo de lenguaje lee millones de palabras y, poco a poco, aprende a predecir la siguiente. No memoriza frases enteras: aprende patrones, regularidades, la forma en que unas palabras llaman a otras. Esa es toda su magia, y también todo su límite.

Para una máquina, una palabra no significa nada hasta que se convierte en números. Cada palabra, o cada letra, recibe un vector: una lista de números que la representa. Las palabras parecidas reciben vectores parecidos, y así el modelo descubre que perro y gato están más cerca entre sí que perro y montaña. Ese mapa de números se llama espacio de representaciones, y es donde vive el significado para la máquina.

El corazón del modelo es la atención. La atención deja que cada palabra mire a todas las demás y decida cuáles importan para entenderla. Cuando lees la frase el perro persiguió al gato porque tenía hambre, sabes que tenía hambre se refiere al perro y no al gato. El modelo aprende a mirar hacia atrás, a poner el foco en la palabra correcta, y resuelve la ambigüedad sin que nadie le diga la regla. Esa mirada repartida, calculada con números, es lo que hace que el lenguaje cobre sentido.

Un modelo grande no tiene una sola mirada, sino muchas. Cada cabeza de atención aprende a fijarse en algo distinto: una vigila la concordancia, otra sigue el hilo del sujeto, otra cuida la puntuación. Varias miradas a la vez ven más que una sola, y por eso los modelos modernos apilan muchas cabezas y muchas capas, una sobre otra, cada una refinando lo que la anterior entendió.

Entrenar un modelo es repetir un ciclo sencillo muchas veces. El modelo lee un trozo de texto y predice la siguiente palabra. Comparamos su predicción con la palabra real y medimos el error. Si se equivoca mucho, el error es grande; si acierta, es pequeño. Después ajustamos los pesos del modelo un poquito, en la dirección que reduce ese error. Predecir, medir, corregir. Una y otra vez, millones de veces, hasta que el modelo aprende la forma del lenguaje.

Cuanto más texto lee y cuanto más entrena, mejor escribe. Al principio genera ruido, letras sueltas sin sentido. Luego empiezan a aparecer sílabas, después palabras, después frases que casi se sostienen. No es que el modelo entienda el mundo: ha aprendido, a fuerza de ejemplos, que después de ciertas letras suelen venir otras, y que ciertas palabras suelen seguir a ciertas palabras.

Generar texto es ir una letra cada vez. El modelo mira lo que lleva escrito, calcula la probabilidad de cada posible siguiente letra y elige una. Luego añade esa letra a lo escrito y vuelve a empezar. Así, paso a paso, construye una frase, un párrafo, una página. La temperatura decide cuánto arriesga: con temperatura baja elige siempre lo más probable y el texto sale seguro y repetitivo; con temperatura alta se atreve con opciones raras y el texto sale variado y sorprendente, a veces a costa de equivocarse.

Lo asombroso es que la arquitectura es siempre la misma. El modelo diminuto que cabe en un cuaderno y el modelo gigante que responde a millones de personas comparten las mismas piezas: números que representan palabras, atención que las relaciona, capas que refinan, y un entrenamiento que corrige el error. La diferencia es de escala, no de naturaleza. Más datos, más parámetros, más entrenamiento. El motor, por dentro, es este mismo.

Por eso conviene construir uno pequeño con las propias manos. Cuando ves al modelo pasar del ruido a las palabras, cuando dibujas el mapa de su atención y compruebas que de verdad mira hacia atrás, cuando juegas con la temperatura y notas cómo cambia su voz, dejas de creer en la magia y empiezas a entender la ingeniería. Y entender es la única forma de usar bien una herramienta tan poderosa.

El lenguaje es un patrón, y un modelo es una máquina que aprende patrones. Esa frase resume casi todo. No hay un alma dentro del modelo, ni una comprensión como la humana. Hay números, hay atención, hay un error que se corrige. Pero de esos ingredientes tan simples, repetidos a gran escala, emerge algo que escribe, traduce, resume y conversa. La curiosidad por cómo ocurre eso es el mejor punto de partida para no perderse en el ruido."""

print('Caracteres en total:', len(texto))
print('---- primeras líneas ----')
print(texto[:280])

## 2. Tokenizar: de letras a números

Una red neuronal solo entiende números. El paso más sencillo posible es trabajar **a nivel de**
**carácter**: hacemos una lista con todos los caracteres distintos que aparecen (el *vocabulario*) y
le damos a cada uno un número. Codificar es cambiar cada letra por su número; descodificar es el
camino de vuelta. Los modelos grandes usan *tokens* de varias letras en vez de caracteres sueltos,
pero la idea es exactamente esta.

In [ ]:
chars = sorted(set(texto))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}   # carácter  -> número
itos = {i: c for i, c in enumerate(chars)}   # número    -> carácter
encode = lambda s: [stoi[c] for c in s]
decode = lambda nums: ''.join(itos[i] for i in nums)

print('Vocabulario (', vocab_size, 'caracteres):')
print(''.join(chars))
print()
ejemplo = 'la atencion'
print(repr(ejemplo), '->', encode(ejemplo))
print(encode(ejemplo), '->', repr(decode(encode(ejemplo))))

## 3. Preparar los datos de entrenamiento

Convertimos todo el texto en un único tensor de números y lo partimos en **entrenamiento** (90 %) y
**validación** (10 %), para vigilar que el modelo aprende de verdad y no solo memoriza.

El modelo aprende a **predecir el siguiente carácter**. Así que cada ejemplo es un trozo de texto (la
entrada `x`) y ese mismo trozo desplazado una posición (la respuesta `y`): para cada letra, la
siguiente. Tomamos esos trozos en lotes (*batches*) de longitud fija `block_size`.

In [ ]:
data = torch.tensor(encode(texto), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

block_size = 64   # cuántos caracteres de contexto ve el modelo a la vez
batch_size = 32   # cuántos trozos procesa en paralelo

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+1+block_size] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print('x (entrada):', tuple(xb.shape), '  y (objetivo):', tuple(yb.shape))
print('Para la entrada', repr(decode(xb[0, :12].tolist())), 'el objetivo es', repr(decode(yb[0, :12].tolist())))
print('es decir: tras cada letra, el modelo debe predecir la siguiente.')

## 4. La pieza central: una cabeza de atención causal

Esta es la idea que hace funcionar al Transformer (capítulos 2 y 3). Cada posición genera tres
vectores: una *query* (qué busco), una *key* (qué ofrezco) y un *value* (qué aporto). El parecido
entre la *query* de una letra y las *keys* de las demás decide **cuánta atención** le presta a cada
una; una `softmax` convierte esos parecidos en pesos que suman 1.

La **máscara causal** es lo que convierte esto en un generador: tapamos el futuro con un triángulo,
de modo que cada posición solo puede mirar hacia atrás, nunca hacia letras que aún no ha escrito.
Guardamos los pesos de atención en `last_att` para poder dibujarlos más adelante.

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.last_att = None
    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        att = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5      # parecido query-key, escalado
        att = att.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # tapar el futuro
        att = F.softmax(att, dim=-1)                             # pesos que suman 1
        self.last_att = att.detach()
        return att @ v                                          # mezcla de values según la atención

## 5. De una mirada a muchas: multi-cabeza, *feed-forward* y el bloque

Una sola cabeza ve un tipo de relación. La **atención multi-cabeza** corre varias en paralelo y junta
sus resultados, así una puede fijarse en la concordancia y otra en la puntuación. Después, una pequeña
red ***feed-forward*** (dos capas con activación GELU) procesa cada posición por separado.

Dos detalles que vimos en el capítulo 4 hacen que esto se pueda apilar en profundidad sin romperse:
las **conexiones residuales** (`x + sublayer(x)`, sumar la entrada a la salida) mantienen vivo el
gradiente, y la **normalización por capas** (*layernorm*) estabiliza los números. Un **bloque** es
atención + *feed-forward*, cada uno con su residual y su *layernorm*. Apilar bloques es lo que da
profundidad al modelo.

In [ ]:
class MultiHead(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(out)

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa = MultiHead(n_head, n_embd // n_head)
        self.ff = FeedForward()
        self.ln1, self.ln2 = nn.LayerNorm(n_embd), nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))   # atención + residual
        x = x + self.ff(self.ln2(x))   # feed-forward + residual
        return x

## 6. El mini-GPT completo

Ya tenemos todas las piezas. El modelo entero es: una tabla de **embeddings de token** (un vector por
carácter del vocabulario), una de **embeddings de posición** (para que sepa el orden), una pila de
**bloques**, una *layernorm* final y una capa que produce las **puntuaciones** (*logits*) de cuál es
el siguiente carácter. Si le pasamos los objetivos, calcula la pérdida con *cross-entropy*, la misma
del capítulo 7.

El método `generate` es el muestreo del capítulo 3: predice la distribución del próximo carácter, la
divide por la **temperatura**, sortea uno, lo añade y repite.

In [ ]:
n_embd = 96    # tamaño de cada vector
n_head = 4     # cabezas de atención por bloque
n_layer = 3    # número de bloques apilados

class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx

model = MiniGPT().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Mini-GPT creado: {n_params:,} parámetros'.replace(',', '.'))
print('(GPT-2 tiene 124 millones; los grandes de hoy, cientos de miles de millones. La arquitectura es la misma.)')

## 7. Generar **antes** de entrenar: ruido

Recién creado, los pesos del modelo son aleatorios: no ha visto el texto, así que no sabe nada del
español. Si le pedimos que genere, saldrá un revoltijo de caracteres. Guárdalo en la cabeza para
compararlo con lo que saldrá después de entrenar.

In [ ]:
contexto = torch.zeros((1, 1), dtype=torch.long, device=device)  # arrancamos del primer carácter
sin_entrenar = decode(model.generate(contexto, 200)[0].tolist())
print(sin_entrenar)

## 8. Entrenar: predecir, medir el error, corregir

Entrenar es el ciclo del capítulo 7, repetido muchas veces: tomar un lote, predecir, medir la pérdida
con *cross-entropy*, calcular gradientes con retropropagación y dar un paso con el optimizador AdamW.
Cada cierto número de pasos medimos la pérdida en entrenamiento y en validación. Si ambas bajan, el
modelo está aprendiendo la forma del español. Verás la curva al final.

Un apunte honesto: como nuestro texto es muy pequeño, el modelo terminará **memorizándolo** en parte.
Por eso verás que la pérdida de *entrenamiento* baja mucho más que la de *validación*: eso es el
**sobreajuste** del capítulo 7, en directo. Aquí es esperable, y hasta útil para verlo de cerca. Con un
corpus grande, las dos curvas bajarían más juntas.

In [ ]:
@torch.no_grad()
def medir_perdida(iters=50):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(iters)
        for k in range(iters):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
max_steps = 2000
historia = []

for step in range(max_steps + 1):
    if step % 300 == 0:
        p = medir_perdida()
        historia.append((step, p['train'], p['val']))
        print(f"paso {step:5d} | pérdida train {p['train']:.3f} | val {p['val']:.3f}")
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print('Listo.')

In [ ]:
pasos = [h[0] for h in historia]
plt.figure(figsize=(8, 4))
plt.plot(pasos, [h[1] for h in historia], 'o-', label='entrenamiento')
plt.plot(pasos, [h[2] for h in historia], 's--', label='validación')
plt.xlabel('paso de entrenamiento'); plt.ylabel('pérdida (cross-entropy)')
plt.title('El modelo aprende: la pérdida baja')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 9. Generar **después** de entrenar: español de verdad

El mismo modelo, los mismos pesos, pero ahora entrenados. Pídele que genere y compara con el ruido de
antes: aparecen palabras, espacios donde tocan, terminaciones españolas. Para tan poco texto y un
modelo tan pequeño, es notable. Con más datos y más capas, esto se convierte en ChatGPT.

In [ ]:
generado = decode(model.generate(contexto, 500)[0].tolist())
print(generado)

## 10. Mirar por dentro: el mapa de atención

Aquí imitamos al *Transformer Explainer*. Le pasamos una frase y dibujamos, para una cabeza, **cuánta**
**atención presta cada carácter a los anteriores**. Cada fila es una posición que mira; cada columna,
una posición mirada. Verás el **triángulo** de la máscara causal (la mitad de arriba a la derecha está
en blanco: nadie mira al futuro) y manchas más oscuras donde una letra se apoya con fuerza en otra.

In [ ]:
frase = 'la atencion mira a las palabras'
ids = torch.tensor([encode(frase)], dtype=torch.long, device=device)
model.eval()
with torch.no_grad():
    model(ids)

att = model.blocks[0].sa.heads[0].last_att[0].cpu()   # bloque 0, cabeza 0
T = len(frase)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(att[:T, :T], cmap='Greys', vmin=0, vmax=1)
ax.set_xticks(range(T)); ax.set_xticklabels(list(frase), fontsize=8)
ax.set_yticks(range(T)); ax.set_yticklabels(list(frase), fontsize=8)
ax.set_xlabel('carácter mirado'); ax.set_ylabel('carácter que mira')
ax.set_title('Atención de una cabeza (bloque 1)')
fig.colorbar(im, ax=ax, fraction=0.046, label='peso de atención')
plt.tight_layout(); plt.show()

Cada cabeza aprende a fijarse en cosas distintas. Dibujemos las cuatro cabezas del primer bloque una
al lado de otra: aunque miran la misma frase, sus patrones no son iguales. Eso es lo que significa que
el modelo mira «desde varios ángulos a la vez».

In [ ]:
fig, axes = plt.subplots(1, n_head, figsize=(4 * n_head, 4))
for h in range(n_head):
    a = model.blocks[0].sa.heads[h].last_att[0].cpu()[:T, :T]
    axes[h].imshow(a, cmap='Greys', vmin=0, vmax=1)
    axes[h].set_title(f'cabeza {h+1}')
    axes[h].set_xticks([]); axes[h].set_yticks([])
plt.suptitle('Las cuatro cabezas miran la misma frase de formas distintas')
plt.tight_layout(); plt.show()

## 11. La temperatura: cómo elige la próxima letra

Cuando el modelo va a escribir, produce una **probabilidad para cada carácter posible**. La
**temperatura** estira o aplana esa distribución antes de sortear: baja (`0.5`) la vuelve más picuda y
el texto sale más seguro y repetitivo; alta (`1.5`) la aplana y el texto sale más variado y arriesgado.
Veámoslo con barras para un mismo contexto.

In [ ]:
prompt = 'la atencion '
ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
with torch.no_grad():
    logits, _ = model(ids)
logits = logits[0, -1].cpu()

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, temp in zip(axes, [0.5, 1.0, 1.5]):
    probs = F.softmax(logits / temp, dim=-1)
    top = torch.topk(probs, 12)
    etiquetas = [repr(itos[i.item()]) for i in top.indices]
    ax.bar(range(12), top.values.numpy(), color='#333333')
    ax.set_xticks(range(12)); ax.set_xticklabels(etiquetas, rotation=45, fontsize=8)
    ax.set_title(f'temperatura = {temp}')
axes[0].set_ylabel('probabilidad')
plt.suptitle(f'¿Qué letra viene tras {prompt!r}? (12 candidatas más probables)')
plt.tight_layout(); plt.show()

In [ ]:
print('El mismo prompt, generado con tres temperaturas:')
for temp in [0.5, 1.0, 1.5]:
    g = decode(model.generate(ids.clone(), 120, temperature=temp)[0].tolist())
    print(f'\n--- temperatura {temp} ---')
    print(g)

## 12. Ahora rómpelo tú

Has construido y entrenado un Transformer de verdad. Para que el conocimiento se asiente, cambia cosas
y vuelve a ejecutar desde la celda correspondiente:

- **Tu propio texto.** Sustituye `texto` en la sección 1 por algo tuyo y más largo. Cuanto más texto, mejor generará.
- **Más o menos profundidad.** Sube `n_layer` a 5 o baja a 1. ¿Cómo cambia la pérdida y el texto?
- **Más entrenamiento.** Sube `max_steps`. ¿Llega un punto en que la validación deja de mejorar (sobreajuste)?
- **Otra cabeza, otro bloque.** En la sección 10, cambia `blocks[0]` por `blocks[1]` o `blocks[2]` y mira si las capas profundas atienden distinto.

La idea de fondo: lo que acabas de construir es, en miniatura, exactamente lo que hace un modelo de
lenguaje grande. La diferencia con ChatGPT no es de naturaleza, sino de escala (más datos, más
parámetros, más entrenamiento) y de un ajuste fino posterior. El motor es este.

---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-28
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*